# India CPI Inflation — Data Analysis Notebook
**Source:** RBI DBIE · MoSPI
**Datasets:** CPI Rural/Urban/Combined (Base 2012=100) + CPI Industrial Workers (Base 2001)

Run cells top-to-bottom. Outputs are saved to `data/processed/` and picked up by the FastAPI backend.

In [3]:
import pandas as pd
import numpy as np
import json
import os
from pathlib import Path

RAW = Path('data/raw')
OUT = Path('data/processed')
OUT.mkdir(exist_ok=True)
print('✅ Libraries loaded')

✅ Libraries loaded


## 1. Load & clean CPI Rural/Urban/Combined

In [4]:
df_raw = pd.read_csv(RAW / 'cpi_rural_urban_combined.csv', header=None)

ruc = df_raw.iloc[:, [1,2,3,4,5,6,7,8,9]].copy()
ruc.columns = ['month','commodity','status',
               'rural_index','rural_inflation',
               'urban_index','urban_inflation',
               'combined_index','combined_inflation']

ruc = ruc[ruc['month'].notna() & (ruc['month'] != 'Month') & ruc['commodity'].notna()]
ruc['date'] = pd.to_datetime(ruc['month'], format='%b-%Y', errors='coerce')
ruc = ruc.dropna(subset=['date'])

for col in ['rural_index','rural_inflation','urban_index','urban_inflation',
            'combined_index','combined_inflation']:
    ruc[col] = pd.to_numeric(ruc[col], errors='coerce')

ruc = ruc.sort_values(['date','commodity']).reset_index(drop=True)
print(f'✅ RUC loaded: {ruc.shape[0]} rows | {ruc["date"].min().date()} → {ruc["date"].max().date()}')
ruc.head(3)

✅ RUC loaded: 4367 rows | 2013-01-01 → 2025-12-01


,month,commodity,status,rural_index,rural_inflation,urban_index,urban_inflation,combined_index,combined_inflation,date
0,JAN-2013,A) General Index,Final,105.1,NaN,104.0,NaN,104.6,NaN,2013-01-01
1,JAN-2013,A.1) Food and beverages,Final,105.5,NaN,105.9,NaN,105.6,NaN,2013-01-01
2,JAN-2013,A.1.1) Cereals and products,Final,107.5,NaN,110.5,NaN,108.4,NaN,2013-01-01


## 2. Load & clean CPI Industrial Workers

In [5]:
df_iw_raw = pd.read_csv(RAW / 'cpi_industrial_workers.csv', header=None)

iw = df_iw_raw.iloc[9:, 1:].copy()
iw.columns = ['date','general_index','food_group','pan_tobacco',
              'fuel_light','housing','clothing','miscellaneous']
iw = iw[iw['date'].notna() & (iw['date'] != 'Month')]
iw['date'] = pd.to_datetime(iw['date'], errors='coerce')
iw = iw.dropna(subset=['date'])
for col in iw.columns[1:]:
    iw[col] = pd.to_numeric(iw[col], errors='coerce')
iw = iw.sort_values('date').reset_index(drop=True)
print(f'✅ IW loaded: {iw.shape[0]} rows | {iw["date"].min().date()} → {iw["date"].max().date()}')
iw.head(3)

✅ IW loaded: 176 rows | 2006-01-31 → 2020-08-31


,date,general_index,food_group,pan_tobacco,fuel_light,housing,clothing,miscellaneous
0,2006-01-31,119.0,119.0,112.0,125.0,123.0,111.0,122.0
1,2006-02-28,119.0,119.0,113.0,126.0,123.0,112.0,123.0
2,2006-03-31,119.0,119.0,114.0,126.0,123.0,112.0,123.0


## 3. Build analysis datasets

In [6]:
# General index only (headline CPI)
general = ruc[ruc['commodity'] == 'A) General Index'].copy()
general = general.sort_values('date')
general['year'] = general['date'].dt.year
general['month_name'] = general['date'].dt.strftime('%b')

# Fill missing YoY inflation from index
general['combined_yoy'] = general['combined_index'].pct_change(12) * 100
general['combined_inflation_final'] = general['combined_inflation'].fillna(
    general['combined_yoy']).round(2)
general['rural_yoy'] = general['rural_index'].pct_change(12) * 100
general['rural_inflation_final'] = general['rural_inflation'].fillna(
    general['rural_yoy']).round(2)
general['urban_yoy'] = general['urban_index'].pct_change(12) * 100
general['urban_inflation_final'] = general['urban_inflation'].fillna(
    general['urban_yoy']).round(2)

print('✅ General index computed')
print(f'Rows: {len(general)} | Years: {general["year"].min()} - {general["year"].max()}')

✅ General index computed
Rows: 156 | Years: 2013 - 2025


In [7]:
# --- TREND DATA (monthly) ---
trend_df = general[['date','rural_index','urban_index','combined_index',
                     'rural_inflation_final','urban_inflation_final',
                     'combined_inflation_final']].dropna(subset=['combined_index'])
trend_df['date_str'] = trend_df['date'].dt.strftime('%Y-%m')

# --- ANNUAL AVG INFLATION ---
annual_df = general.groupby('year').agg(
    avg_combined=('combined_inflation_final','mean'),
    avg_rural=('rural_inflation_final','mean'),
    avg_urban=('urban_inflation_final','mean')
).round(2).reset_index()
annual_df = annual_df[annual_df['year'] >= 2014]

# --- HEATMAP (year x month) ---
heatmap_df = general[general['year'] >= 2014].copy()
month_order = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
heatmap_pivot = heatmap_df.pivot_table(
    index='year', columns='month_name',
    values='combined_inflation_final', aggfunc='mean'
).round(2)
heatmap_pivot = heatmap_pivot.reindex(
    columns=[m for m in month_order if m in heatmap_pivot.columns])

# --- CATEGORY BREAKDOWN (latest month) ---
latest_month = ruc['date'].max()
latest_data = ruc[ruc['date'] == latest_month].copy()
top_cats = [
    'A) General Index', 'A.1) Food and beverages',
    'A.3) Clothing and footwear', 'A.4) Housing',
    'A.5) Fuel and light', 'A.6) Miscellaneous',
    'B) Consumer Food Price Index'
]
cat_df = latest_data[latest_data['commodity'].isin(top_cats)][
    ['commodity','combined_index','combined_inflation']].copy()
cat_df.columns = ['category','index_val','inflation']
cat_df['category'] = cat_df['category'].str.replace(r'^[A-Z][0-9.]*\) ', '', regex=True)
cat_df['inflation'] = pd.to_numeric(cat_df['inflation'], errors='coerce').round(2)

# --- SUMMARY STATS ---
latest_general = general[general['date'] == general['date'].max()].iloc[0]
peak_year = annual_df.loc[annual_df['avg_combined'].idxmax()]

print('✅ All datasets computed')
print(f'Latest month: {latest_month.strftime("%B %Y")}')
print(f'Latest combined inflation: {latest_general["combined_inflation_final"]}%')
print(f'Peak year: {int(peak_year["year"])} @ {peak_year["avg_combined"]}%')

✅ All datasets computed
Latest month: December 2025
Latest combined inflation: 1.33%
Peak year: 2014 @ 6.71%


## 4. Export to JSON (picked up by FastAPI)

In [8]:
def safe_float(x):
    if pd.isna(x): return None
    return round(float(x), 2)

# Trend
trend_out = []
for _, row in trend_df.iterrows():
    trend_out.append({
        'date': row['date_str'],
        'rural_index': safe_float(row['rural_index']),
        'urban_index': safe_float(row['urban_index']),
        'combined_index': safe_float(row['combined_index']),
        'rural_inflation': safe_float(row['rural_inflation_final']),
        'urban_inflation': safe_float(row['urban_inflation_final']),
        'combined_inflation': safe_float(row['combined_inflation_final'])
    })

# Annual
annual_out = [
    {'year': int(r['year']),
     'avg_combined': safe_float(r['avg_combined']),
     'avg_rural': safe_float(r['avg_rural']),
     'avg_urban': safe_float(r['avg_urban'])}
    for _, r in annual_df.iterrows()
]

# Heatmap
heatmap_out = []
for year, row in heatmap_pivot.iterrows():
    for month, val in row.items():
        if not pd.isna(val):
            heatmap_out.append({'year': int(year), 'month': month, 'inflation': round(float(val), 2)})

# Categories
cat_out = [
    {'category': r['category'],
     'index_val': safe_float(r['index_val']),
     'inflation': safe_float(r['inflation'])}
    for _, r in cat_df.dropna(subset=['inflation']).iterrows()
]

# Summary
food_row = latest_data[latest_data['commodity'] == 'A.1) Food and beverages']
food_inf = safe_float(food_row['combined_inflation'].values[0]) if len(food_row) else None

summary_out = {
    'latest_month': latest_month.strftime('%B %Y'),
    'latest_combined_inflation': safe_float(latest_general['combined_inflation_final']),
    'latest_rural_inflation': safe_float(latest_general['rural_inflation_final']),
    'latest_urban_inflation': safe_float(latest_general['urban_inflation_final']),
    'latest_food_inflation': food_inf,
    'peak_year': int(peak_year['year']),
    'peak_inflation': safe_float(peak_year['avg_combined']),
    'rural_urban_gap': round(
        (safe_float(latest_general['rural_inflation_final']) or 0) -
        (safe_float(latest_general['urban_inflation_final']) or 0), 2)
}

# Write all files
files = {
    'trend.json': trend_out,
    'annual.json': annual_out,
    'heatmap.json': heatmap_out,
    'categories.json': cat_out,
    'summary.json': summary_out
}

for fname, data in files.items():
    with open(OUT / fname, 'w') as f:
        json.dump(data, f, indent=2)
    print(f'✅ Saved {fname}')

print(f'\n🎉 All done! Backend will serve from data/processed/')

✅ Saved trend.json
✅ Saved annual.json
✅ Saved heatmap.json
✅ Saved categories.json
✅ Saved summary.json

🎉 All done! Backend will serve from data/processed/
